In [1]:
# =================================================================================================
# NHANES 2017-2018
# SLEEP DISORDERS QUESTIONNAIRE - SLQ_J
#
# INPUT:
#   SLQ_J.xpt
#
# OUTPUT:
#   SLQ_J_corrected.csv
#
#
# COMPONENT:
#   Questionnaire
#
# SUBCOMPONENT:
#   Sleep Disorders
#
#
# ACTUAL ATTACHED SOURCE
# -------------------------------------------------------------------------------------------------
#
# Rows                         = 6,161
# Columns                      = 11
#
# Variables:
#
#   SEQN
#   SLQ300
#   SLQ310
#   SLD012
#   SLQ320
#   SLQ330
#   SLD013
#   SLQ030
#   SLQ040
#   SLQ050
#   SLQ120
#
#
# SEQN range                   = 93705-102956
# Missing SEQN                 = 0
# Duplicate SEQN               = 0
#
#
# VALIDATED SAS XPORT TINY-ZERO ARTIFACTS
# -------------------------------------------------------------------------------------------------
#
# SLQ030      1,688
# SLQ040      4,474
# SLQ120      1,028
#
# TOTAL        7,190
#
#
# These are legitimate:
#
#   0 = Never
#
# values according to the CDC codebook.
#
#
# CHARACTER VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SLQ300   Usual sleep time - weekdays/workdays
# SLQ310   Usual wake time  - weekdays/workdays
# SLQ320   Usual sleep time - weekends/non-workdays
# SLQ330   Usual wake time  - weekends/non-workdays
#
# CDC defines these as character HH:MM variables.
#
# They MUST remain exact character text.
#
#
# GENUINE DECIMAL / DERIVED VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SLD012   Sleep hours - weekdays/workdays
# SLD013   Sleep hours - weekends/non-workdays
#
# CDC derives these from sleep/wake times and rounds to the
# nearest half-hour.
#
# Therefore values such as:
#
#   6.5
#   7.5
#   8.5
#
# are genuine and MUST NOT be integerized.
#
#
# PROVEN INTEGRAL/CODE VARIABLES
# -------------------------------------------------------------------------------------------------
#
# SEQN
# SLQ030
# SLQ040
# SLQ050
# SLQ120
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# 1. Restore ONLY validated tiny XPORT zeros.
#
# 2. Never globally replace tiny values.
#
# 3. Preserve SLD012 and SLD013 decimal values.
#
# 4. Preserve SLQ300 / SLQ310 / SLQ320 / SLQ330 text exactly.
#
# 5. Remove ".0" only from proven integral/code fields.
#
# 6. Never round source data.
#
# 7. Never use float_format=.
#
# 8. Preserve missing positions.
#
# 9. Verify CDC distributions.
#
# 10. Verify XPT -> CSV numeric round-trip exactly.
#
# 11. Verify character variables text-for-text after CSV export.
#
#
# GENERAL NHANES PIPELINE RULE
# -------------------------------------------------------------------------------------------------
#
# Genuine decimal values in Dietary / Lab / Examination / Demo /
# Questionnaire data must remain AS IS.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


QUESTIONNAIRE_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)


XPT_FILE = (
    QUESTIONNAIRE_DIR
    / "SLQ_J.xpt"
)


CSV_FILE = (
    QUESTIONNAIRE_DIR
    / "SLQ_J_corrected.csv"
)


# =================================================================================================
# 3. VERIFY INPUT FILE EXISTS
# =================================================================================================

if not XPT_FILE.exists():

    raise FileNotFoundError(
        f"SLQ_J XPT file not found:\n{XPT_FILE}"
    )


# =================================================================================================
# 4. READ ORIGINAL XPT
# =================================================================================================

df = pd.read_sas(
    XPT_FILE,
    format="xport",
    encoding="latin1"
)


#
# Keep completely untouched decoded source for fidelity comparisons.
#

source_df = df.copy()


print("=" * 110)

print(
    "NHANES 2017-2018 SLQ_J"
)

print(
    "SLEEP DISORDERS QUESTIONNAIRE"
)

print("=" * 110)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Variables:"
)


print(
    df.columns.tolist()
)


print(
    "\nPandas source dtypes:"
)


print(
    df.dtypes
)


# =================================================================================================
# 5. EXPECTED SOURCE STRUCTURE
# =================================================================================================

EXPECTED_COLUMNS = [

    "SEQN",

    "SLQ300",

    "SLQ310",

    "SLD012",

    "SLQ320",

    "SLQ330",

    "SLD013",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


EXPECTED_ROWS = 6161


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SLQ_J column structure differs from validated source.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected SLQ_J row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


print(
    "PASS: SLQ_J structure = 6,161 rows x 11 columns."
)


# =================================================================================================
# 6. FIELD CLASSIFICATION
# =================================================================================================

CHARACTER_COLUMNS = [

    "SLQ300",

    "SLQ310",

    "SLQ320",

    "SLQ330"

]


DECIMAL_COLUMNS = [

    "SLD012",

    "SLD013"

]


INTEGER_COLUMNS = [

    "SEQN",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


NUMERIC_COLUMNS = [

    "SEQN",

    "SLD012",

    "SLD013",

    "SLQ030",

    "SLQ040",

    "SLQ050",

    "SLQ120"

]


print(
    "\n--- FIELD TYPE PLAN ---"
)


print(
    "Character fields:",
    CHARACTER_COLUMNS
)


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "Integral/code fields:",
    INTEGER_COLUMNS
)


# =================================================================================================
# 7. CHARACTER TEXT HELPER
# =================================================================================================

#
# Do NOT depend on pandas saying dtype == object.
#
# Depending on pandas version, character data can appear as:
#
#   object
#   string
#   StringDtype
#
# Fidelity is determined from actual content.
#

def text_view(series):

    return (
        series
        .astype("string")
        .fillna("")
    )


# =================================================================================================
# 8. VERIFY NUMERIC FIELDS
# =================================================================================================

for col in NUMERIC_COLUMNS:

    if not pd.api.types.is_numeric_dtype(
        df[col]
    ):

        raise ValueError(
            f"{col} expected to load numerically, "
            f"but pandas loaded dtype={df[col].dtype}."
        )


print(
    "PASS: Numeric source fields validated."
)


# =================================================================================================
# 9. VERIFY CHARACTER FIELD CONTENT
# =================================================================================================

#
# Allowed:
#
#   HH:MM
#   77777
#   99999
#   blank
#

for col in CHARACTER_COLUMNS:

    text = text_view(
        df[col]
    )


    time_mask = text.str.fullmatch(
        r"\d{2}:\d{2}",
        na=False
    )


    special_mask = text.isin(
        [
            "77777",
            "99999",
            ""
        ]
    )


    invalid_mask = ~(
        time_mask
        |
        special_mask
    )


    invalid_count = int(
        invalid_mask.sum()
    )


    if invalid_count != 0:

        print(
            f"\nUnexpected {col} examples:"
        )


        print(
            text.loc[
                invalid_mask
            ]
            .head(20)
            .tolist()
        )


        raise ValueError(
            f"{col} contains {invalid_count:,} "
            "unexpected text values."
        )


    #
    # Validate HH:MM clock values.
    #

    times = text.loc[
        time_mask
    ]


    hours = pd.to_numeric(

        times.str.slice(
            0,
            2
        ),

        errors="raise"

    )


    minutes = pd.to_numeric(

        times.str.slice(
            3,
            5
        ),

        errors="raise"

    )


    invalid_clock = (

        ~hours.between(
            0,
            23
        )

        |

        ~minutes.between(
            0,
            59
        )

    )


    if invalid_clock.any():

        raise ValueError(
            f"{col} contains invalid HH:MM clock values."
        )


    print(
        f"PASS: {col} character content validated."
    )


# =================================================================================================
# 10. SEQN SOURCE CHECK
# =================================================================================================

seqn_min = int(
    df["SEQN"].min()
)


seqn_max = int(
    df["SEQN"].max()
)


seqn_missing = int(

    df["SEQN"]
    .isna()
    .sum()

)


seqn_duplicates = int(

    df["SEQN"]
    .duplicated()
    .sum()

)


print(
    "\n--- SEQN SOURCE CHECK ---"
)


print(
    "Minimum:",
    seqn_min
)


print(
    "Maximum:",
    seqn_max
)


print(
    "Missing:",
    seqn_missing
)


print(
    "Duplicates:",
    seqn_duplicates
)


if seqn_min != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if seqn_max != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if seqn_missing != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if seqn_duplicates != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN range 93705-102956 validated."
)


# =================================================================================================
# 11. DEFINE KNOWN SAS XPORT TINY ZERO VALUE
# =================================================================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


# =================================================================================================
# 12. DISCOVER TINY VALUES
# =================================================================================================

tiny_counts = (

    df[NUMERIC_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPORT TINY-ZERO CHECK ---"
)


print(
    tiny_counts.to_string()
)


print(
    "\nTotal tiny XPORT values:",
    f"{tiny_total:,}"
)


# =================================================================================================
# 13. VALIDATED TINY-ZERO PATTERN
# =================================================================================================

#
# Official CDC:
#
# SLQ030 = 0 / Never = 1,688
#
# SLQ040 = 0 / Never = 4,474
#
# SLQ120 = 0 / Never = 1,028
#

EXPECTED_TINY_COUNTS = {

    "SLQ030": 1688,

    "SLQ040": 4474,

    "SLQ120": 1028

}


actual_tiny_counts = (

    tiny_counts[
        tiny_counts > 0
    ]

    .astype(int)

    .to_dict()

)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: SLQ_J tiny-value pattern differs "
        "from validated source.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 7190:

    raise ValueError(
        "Unexpected total tiny-value count.\n"
        "Expected: 7,190\n"
        f"Actual:   {tiny_total:,}"
    )


print(
    "PASS: Exact 7,190-value tiny-zero pattern validated."
)


# =================================================================================================
# 14. ORDINARY ZERO CHECK BEFORE RESTORATION
# =================================================================================================

ordinary_zero_counts = (

    df[NUMERIC_COLUMNS]

    .eq(0)

    .sum()

)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORDINARY ZERO CHECK BEFORE RESTORATION ---"
)


print(
    ordinary_zero_counts.to_string()
)


print(
    "\nTotal ordinary numeric zeros:",
    ordinary_zero_total
)


if ordinary_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before XPORT correction."
    )


print(
    "PASS: Legitimate zero categories are represented "
    "by the SAS XPORT tiny value."
)


# =================================================================================================
# 15. BUILD EXPECTED CORRECTED SOURCE
# =================================================================================================

#
# expected_df represents the ONLY scientifically approved
# modification:
#
# tiny validated zero -> actual numeric 0.
#

expected_df = (
    source_df.copy()
)


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[

        expected_df[col]
        .eq(TINY_VALUE),

        col

    ] = 0


# =================================================================================================
# 16. RESTORE ONLY VALIDATED TRUE ZEROS
# =================================================================================================

correction_counts = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny-value count differs.\n"
            f"Expected: {expected_count:,}\n"
            f"Actual:   {actual_count:,}"
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_counts[
        col
    ] = actual_count


total_corrections = sum(
    correction_counts.values()
)


print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


for col, count in correction_counts.items():

    print(
        f"{col}: {count:,}"
    )


print(
    "TOTAL:",
    f"{total_corrections:,}"
)


if total_corrections != 7190:

    raise ValueError(
        "Incorrect zero-restoration total."
    )


print(
    "PASS: 7,190 validated zeros restored."
)


# =================================================================================================
# 17. VERIFY NO TINY VALUE REMAINS
# =================================================================================================

remaining_tiny = int(

    df[NUMERIC_COLUMNS]

    .eq(TINY_VALUE)

    .sum()

    .sum()

)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# =================================================================================================
# 18. VERIFY FINAL TRUE-ZERO PATTERN
# =================================================================================================

corrected_zero_counts = {}


for col in NUMERIC_COLUMNS:

    count = int(

        df[col]
        .eq(0)
        .sum()

    )


    if count > 0:

        corrected_zero_counts[
            col
        ] = count


print(
    "\n--- CORRECTED TRUE-ZERO DISTRIBUTION ---"
)


print(
    corrected_zero_counts
)


if corrected_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Corrected zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{corrected_zero_counts}"
    )


print(
    "PASS: Corrected zero distribution validated."
)


# =================================================================================================
# 19. CDC CHARACTER FREQUENCY CHECK FUNCTION
# =================================================================================================

def validate_time_field(
    dataset,
    column,
    expected_recorded,
    expected_refused,
    expected_dontknow,
    expected_missing
):

    text = text_view(
        dataset[column]
    )


    recorded = int(

        text.str.fullmatch(
            r"\d{2}:\d{2}",
            na=False
        )
        .sum()

    )


    refused = int(

        text.eq(
            "77777"
        )
        .sum()

    )


    dontknow = int(

        text.eq(
            "99999"
        )
        .sum()

    )


    missing = int(

        text.eq("")
        .sum()

    )


    print(
        f"\n--- CDC CHECK: {column} ---"
    )


    print(
        "Recorded HH:MM:",
        f"{recorded:,}"
    )


    print(
        "Refused:",
        refused
    )


    print(
        "Don't know:",
        dontknow
    )


    print(
        "Missing:",
        missing
    )


    if recorded != expected_recorded:

        raise ValueError(
            f"{column}: recorded-time count differs from CDC."
        )


    if refused != expected_refused:

        raise ValueError(
            f"{column}: Refused count differs from CDC."
        )


    if dontknow != expected_dontknow:

        raise ValueError(
            f"{column}: Don't know count differs from CDC."
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column}: missing count differs from CDC."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 20. CDC CHARACTER FIELD CHECKS
# =================================================================================================

validate_time_field(

    df,

    "SLQ300",

    expected_recorded=6087,

    expected_refused=1,

    expected_dontknow=36,

    expected_missing=37

)


validate_time_field(

    df,

    "SLQ310",

    expected_recorded=6100,

    expected_refused=1,

    expected_dontknow=21,

    expected_missing=39

)


validate_time_field(

    df,

    "SLQ320",

    expected_recorded=6090,

    expected_refused=1,

    expected_dontknow=39,

    expected_missing=31

)


validate_time_field(

    df,

    "SLQ330",

    expected_recorded=6102,

    expected_refused=1,

    expected_dontknow=26,

    expected_missing=32

)


print(
    "\nPASS: All four character sleep/wake variables "
    "match CDC."
)


# =================================================================================================
# 21. CDC CHECK - SLD012
# =================================================================================================

#
# CDC:
#
# 3-13.5 = 6,073
# 2      =    27
# 14     =    13
# Missing=    48
#
# Total populated = 6,113
#

sld012 = (
    df["SLD012"]
)


sld012_middle = int(

    sld012
    .between(
        3,
        13.5
    )
    .sum()

)


sld012_two = int(

    sld012
    .eq(2)
    .sum()

)


sld012_fourteen = int(

    sld012
    .eq(14)
    .sum()

)


sld012_missing = int(

    sld012
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: SLD012 ---"
)


print(
    "3-13.5:",
    f"{sld012_middle:,}"
)


print(
    "<3 code (2):",
    sld012_two
)


print(
    "14+ code (14):",
    sld012_fourteen
)


print(
    "Missing:",
    sld012_missing
)


if sld012_middle != 6073:

    raise ValueError(
        "SLD012 3-13.5 count differs from CDC."
    )


if sld012_two != 27:

    raise ValueError(
        "SLD012=2 count differs from CDC."
    )


if sld012_fourteen != 13:

    raise ValueError(
        "SLD012=14 count differs from CDC."
    )


if sld012_missing != 48:

    raise ValueError(
        "SLD012 missing count differs from CDC."
    )


if int(
    sld012.notna().sum()
) != 6113:

    raise ValueError(
        "SLD012 populated count differs."
    )


print(
    "PASS: SLD012 matches CDC."
)


# =================================================================================================
# 22. CDC CHECK - SLD013
# =================================================================================================

#
# CDC:
#
# 3-13.5 = 6,061
# 2      =    12
# 14     =    31
# Missing=    57
#
# Total populated = 6,104
#

sld013 = (
    df["SLD013"]
)


sld013_middle = int(

    sld013
    .between(
        3,
        13.5
    )
    .sum()

)


sld013_two = int(

    sld013
    .eq(2)
    .sum()

)


sld013_fourteen = int(

    sld013
    .eq(14)
    .sum()

)


sld013_missing = int(

    sld013
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: SLD013 ---"
)


print(
    "3-13.5:",
    f"{sld013_middle:,}"
)


print(
    "<3 code (2):",
    sld013_two
)


print(
    "14+ code (14):",
    sld013_fourteen
)


print(
    "Missing:",
    sld013_missing
)


if sld013_middle != 6061:

    raise ValueError(
        "SLD013 3-13.5 count differs from CDC."
    )


if sld013_two != 12:

    raise ValueError(
        "SLD013=2 count differs from CDC."
    )


if sld013_fourteen != 31:

    raise ValueError(
        "SLD013=14 count differs from CDC."
    )


if sld013_missing != 57:

    raise ValueError(
        "SLD013 missing count differs from CDC."
    )


if int(
    sld013.notna().sum()
) != 6104:

    raise ValueError(
        "SLD013 populated count differs."
    )


print(
    "PASS: SLD013 matches CDC."
)


# =================================================================================================
# 23. VERIFY HALF-HOUR INCREMENTS
# =================================================================================================

for col in DECIMAL_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    half_hour_check = np.isclose(

        values * 2,

        np.round(
            values * 2
        ),

        rtol=0,

        atol=1e-12

    )


    if not half_hour_check.all():

        raise ValueError(
            f"{col} contains values outside "
            "half-hour increments."
        )


print(
    "PASS: SLD012 and SLD013 use valid half-hour increments."
)


# =================================================================================================
# 24. CDC CHECK - SLQ030
# =================================================================================================

EXPECTED_SLQ030 = {

    0.0: 1688,

    1.0: 1370,

    2.0: 1068,

    3.0: 1577,

    7.0: 7,

    9.0: 451

}


actual_slq030 = (

    df["SLQ030"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


print(
    "\n--- CDC CHECK: SLQ030 ---"
)


print(
    actual_slq030
)


if actual_slq030 != EXPECTED_SLQ030:

    raise ValueError(
        "SLQ030 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ030}\n\n"
        f"Actual:\n{actual_slq030}"
    )


if int(
    df["SLQ030"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ030 unexpected missing values."
    )


print(
    "PASS: SLQ030 matches CDC exactly."
)


# =================================================================================================
# 25. CDC CHECK - SLQ040
# =================================================================================================

EXPECTED_SLQ040 = {

    0.0: 4474,

    1.0: 699,

    2.0: 361,

    3.0: 280,

    7.0: 3,

    9.0: 344

}


actual_slq040 = (

    df["SLQ040"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


print(
    "\n--- CDC CHECK: SLQ040 ---"
)


print(
    actual_slq040
)


if actual_slq040 != EXPECTED_SLQ040:

    raise ValueError(
        "SLQ040 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ040}\n\n"
        f"Actual:\n{actual_slq040}"
    )


if int(
    df["SLQ040"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ040 unexpected missing values."
    )


print(
    "PASS: SLQ040 matches CDC exactly."
)


# =================================================================================================
# 26. CDC CHECK - SLQ050
# =================================================================================================

EXPECTED_SLQ050 = {

    1.0: 1621,

    2.0: 4535,

    9.0: 5

}


actual_slq050 = (

    df["SLQ050"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


print(
    "\n--- CDC CHECK: SLQ050 ---"
)


print(
    actual_slq050
)


if actual_slq050 != EXPECTED_SLQ050:

    raise ValueError(
        "SLQ050 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ050}\n\n"
        f"Actual:\n{actual_slq050}"
    )


if int(
    df["SLQ050"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ050 unexpected missing values."
    )


print(
    "PASS: SLQ050 matches CDC exactly."
)


# =================================================================================================
# 27. CDC CHECK - SLQ120
# =================================================================================================

EXPECTED_SLQ120 = {

    0.0: 1028,

    1.0: 1468,

    2.0: 2059,

    3.0: 1066,

    4.0: 528,

    9.0: 12

}


actual_slq120 = (

    df["SLQ120"]

    .value_counts(
        dropna=True
    )

    .sort_index()

    .to_dict()

)


print(
    "\n--- CDC CHECK: SLQ120 ---"
)


print(
    actual_slq120
)


if actual_slq120 != EXPECTED_SLQ120:

    raise ValueError(
        "SLQ120 distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_SLQ120}\n\n"
        f"Actual:\n{actual_slq120}"
    )


if int(
    df["SLQ120"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SLQ120 unexpected missing values."
    )


print(
    "PASS: SLQ120 matches CDC exactly."
)


# =================================================================================================
# 28. FRACTIONAL VALUE DISCOVERY
# =================================================================================================

#
# Diagnostic only.
#
# np.round() is NOT written back to the data.
#

fractional_counts = {}


for col in NUMERIC_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) == 0:

        count = 0


    else:

        count = int(

            (
                ~np.isclose(

                    values,

                    np.round(
                        values
                    ),

                    rtol=0,

                    atol=1e-12

                )
            )

            .sum()

        )


    fractional_counts[
        col
    ] = count


print(
    "\n--- FRACTIONAL VALUE DISCOVERY ---"
)


for col, count in fractional_counts.items():

    print(
        f"{col}: {count:,}"
    )


#
# Actual uploaded SLQ_J:
#
# SLD012 fractional observations = 1,833
# SLD013 fractional observations = 1,337
#

EXPECTED_FRACTIONAL_COUNTS = {

    "SEQN": 0,

    "SLD012": 1833,

    "SLD013": 1337,

    "SLQ030": 0,

    "SLQ040": 0,

    "SLQ050": 0,

    "SLQ120": 0

}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Fractional-value pattern differs from "
        "the validated attached SLQ_J source.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: SLD012 and SLD013 are the only "
    "genuine fractional numeric fields."
)


# =================================================================================================
# 29. VERIFY ONLY VALIDATED NUMERIC VALUES CHANGED
# =================================================================================================

source_change_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        df[col]

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        source_change_failures.append(
            col
        )


if source_change_failures:

    raise ValueError(
        "Unexpected numeric source-value changes:\n"
        + str(
            source_change_failures
        )
    )


print(
    "PASS: Only validated tiny-zero values changed."
)


# =================================================================================================
# 30. CHARACTER SOURCE FIDELITY BEFORE EXPORT
# =================================================================================================

for col in CHARACTER_COLUMNS:

    original_text = text_view(
        source_df[col]
    )


    current_text = text_view(
        df[col]
    )


    if not original_text.equals(
        current_text
    ):

        raise ValueError(
            f"{col} character values changed during cleaning."
        )


print(
    "PASS: All four sleep/wake character fields remain unchanged."
)


# =================================================================================================
# 31. MISSING-POSITION FIDELITY
# =================================================================================================

#
# Numeric fields.
#

for col in NUMERIC_COLUMNS:

    source_missing = (

        source_df[col]
        .isna()
        .to_numpy()

    )


    current_missing = (

        df[col]
        .isna()
        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        current_missing

    ):

        raise ValueError(
            f"{col}: numeric missing positions changed."
        )


#
# Character fields.
#

for col in CHARACTER_COLUMNS:

    source_missing = (

        text_view(
            source_df[col]
        )

        .eq("")

        .to_numpy()

    )


    current_missing = (

        text_view(
            df[col]
        )

        .eq("")

        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        current_missing

    ):

        raise ValueError(
            f"{col}: character missing positions changed."
        )


print(
    "PASS: Missing positions preserved exactly."
)


# =================================================================================================
# 32. CONVERT ONLY PROVEN INTEGRAL/CODE FIELDS TO Int64
# =================================================================================================

#
# This fixes:
#
#   93705.0 -> 93705
#
#   1.0 -> 1
#
#
# DO NOT convert:
#
#   SLD012
#   SLD013
#
#
# DO NOT touch:
#
#   SLQ300
#   SLQ310
#   SLQ320
#   SLQ330
#

for col in INTEGER_COLUMNS:

    values = (

        df[col]

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    if len(values) > 0:

        integral_mask = np.isclose(

            values,

            np.round(
                values
            ),

            rtol=0,

            atol=1e-12

        )


        if not integral_mask.all():

            raise ValueError(
                f"{col} contains unexpected fractional values."
            )


    df[col] = (

        df[col]

        .astype("Int64")

    )


print(
    "PASS: Only proven integral/code fields converted to Int64."
)


# =================================================================================================
# 33. PRE-EXPORT NUMERIC FIDELITY
# =================================================================================================

pre_export_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    current_values = (

        pd.to_numeric(
            df[col],
            errors="coerce"
        )

        .to_numpy(
            dtype=float,
            na_value=np.nan
        )

    )


    same = np.allclose(

        expected_values,

        current_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Pre-export numeric fidelity failures:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export numeric fidelity."
)


# =================================================================================================
# 34. PRE-EXPORT CHARACTER FIDELITY
# =================================================================================================

for col in CHARACTER_COLUMNS:

    if not text_view(
        df[col]
    ).equals(
        text_view(
            source_df[col]
        )
    ):

        raise ValueError(
            f"{col} changed before CSV export."
        )


print(
    "PASS: Pre-export character fidelity."
)


# =================================================================================================
# 35. EXPORT CORRECTED CSV
# =================================================================================================

#
# CRITICAL:
#
# NO:
#
#   float_format=
#
# NO:
#
#   round()
#
# NO integer conversion of:
#
#   SLD012
#   SLD013
#

df.to_csv(

    CSV_FILE,

    index=False,

    na_rep=""

)


print(
    "\nCorrected CSV created:"
)


print(
    CSV_FILE
)


# =================================================================================================
# 36. RE-READ FINAL CSV AS RAW TEXT
# =================================================================================================

raw_csv = pd.read_csv(

    CSV_FILE,

    dtype=str,

    keep_default_na=False

)


if raw_csv.shape != (

    EXPECTED_ROWS,

    len(EXPECTED_COLUMNS)

):

    raise ValueError(
        "Final CSV dimensions changed."
    )


if (
    raw_csv.columns.tolist()
    !=
    EXPECTED_COLUMNS
):

    raise ValueError(
        "Final CSV column structure/order changed."
    )


print(
    "PASS: Final CSV structure preserved."
)


# =================================================================================================
# 37. CHARACTER TEXT FIDELITY AFTER EXPORT
# =================================================================================================

for col in CHARACTER_COLUMNS:

    source_text = (

        text_view(
            source_df[col]
        )

        .to_numpy(
            dtype=object
        )

    )


    final_text = (

        raw_csv[col]

        .to_numpy(
            dtype=object
        )

    )


    if not np.array_equal(

        source_text,

        final_text

    ):

        mismatch_mask = (

            source_text
            !=
            final_text

        )


        mismatch_count = int(
            mismatch_mask.sum()
        )


        raise ValueError(
            f"{col} changed during CSV export. "
            f"Mismatches={mismatch_count:,}"
        )


print(
    "PASS: SLQ300 / SLQ310 / SLQ320 / SLQ330 "
    "preserved text-for-text."
)


# =================================================================================================
# 38. EXACT NUMERIC XPT -> CSV ROUND-TRIP
# =================================================================================================

#
# Numeric source fidelity uses:
#
#   rtol=0
#   atol=0
#
# after approved zero restoration.
#

roundtrip_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (

        expected_df[col]

        .to_numpy(
            dtype=float
        )

    )


    final_values = (

        pd.to_numeric(

            raw_csv[col]
            .where(
                raw_csv[col] != "",
                np.nan
            ),

            errors="coerce"

        )

        .to_numpy(
            dtype=float
        )

    )


    same = np.allclose(

        expected_values,

        final_values,

        rtol=0,

        atol=0,

        equal_nan=True

    )


    if not same:

        roundtrip_failures.append(
            col
        )


if roundtrip_failures:

    raise ValueError(
        "XPT -> CSV exact numeric fidelity failures:\n"
        + str(
            roundtrip_failures
        )
    )


print(
    "PASS: All numeric fields round-trip exactly."
)


# =================================================================================================
# 39. CHECK UNWANTED ".0"
# =================================================================================================

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    bad_mask = (

        raw_csv[col]
        .ne("")

        &

        raw_csv[col]
        .str.endswith(".0")

    )


    bad_count = int(
        bad_mask.sum()
    )


    if bad_count > 0:

        dot_zero_errors[
            col
        ] = bad_count


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' formatting remains:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in integral/code fields."
)


# =================================================================================================
# 40. FINAL SEQN CHECK
# =================================================================================================

if (
    raw_csv["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    raw_csv["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~raw_csv["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN."
    )


final_seqn = pd.to_numeric(

    raw_csv["SEQN"],

    errors="raise"

)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN differs."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN differs."
    )


print(
    "PASS: Final SEQN clean."
)


# =================================================================================================
# 41. FINAL DECIMAL-PRESERVATION CHECK
# =================================================================================================

for col, expected_fractional in {

    "SLD012": 1833,

    "SLD013": 1337

}.items():

    final_values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="raise"

    )


    values = (

        final_values

        .dropna()

        .to_numpy(
            dtype=float
        )

    )


    fractional_count = int(

        (
            ~np.isclose(

                values,

                np.round(
                    values
                ),

                rtol=0,

                atol=1e-12

            )
        )

        .sum()

    )


    print(
        f"{col} final fractional observations:",
        f"{fractional_count:,}"
    )


    if fractional_count != expected_fractional:

        raise ValueError(
            f"{col} fractional values changed."
        )


print(
    "PASS: SLD012 and SLD013 decimals preserved."
)


# =================================================================================================
# 42. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_total = 0


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    final_tiny_total += int(

        values
        .eq(TINY_VALUE)
        .sum()

    )


if final_tiny_total != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# =================================================================================================
# 43. FINAL TRUE-ZERO DISTRIBUTION
# =================================================================================================

final_zero_counts = {}


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(

        raw_csv[col]
        .where(
            raw_csv[col] != "",
            np.nan
        ),

        errors="coerce"

    )


    count = int(

        values
        .eq(0)
        .sum()

    )


    if count > 0:

        final_zero_counts[
            col
        ] = count


print(
    "\n--- FINAL TRUE-ZERO DISTRIBUTION ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero distribution differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Exactly 7,190 validated true zeros remain."
)


# =================================================================================================
# 44. FINAL SCIENTIFIC-NOTATION CHECK
# =================================================================================================

scientific_notation_fields = {}


for col in NUMERIC_COLUMNS:

    count = int(

        raw_csv[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


if scientific_notation_fields:

    raise ValueError(
        "Unexpected scientific notation remains:\n"
        + str(
            scientific_notation_fields
        )
    )


print(
    "PASS: No unexpected scientific notation."
)


# =================================================================================================
# 45. FINAL MISSING-POSITION FIDELITY
# =================================================================================================

#
# Numeric fields.
#

for col in NUMERIC_COLUMNS:

    source_missing = (

        source_df[col]
        .isna()
        .to_numpy()

    )


    final_missing = (

        raw_csv[col]
        .eq("")
        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        final_missing

    ):

        raise ValueError(
            f"{col}: final missing positions differ "
            "from source XPT."
        )


#
# Character fields.
#

for col in CHARACTER_COLUMNS:

    source_missing = (

        text_view(
            source_df[col]
        )

        .eq("")

        .to_numpy()

    )


    final_missing = (

        raw_csv[col]
        .eq("")
        .to_numpy()

    )


    if not np.array_equal(

        source_missing,

        final_missing

    ):

        raise ValueError(
            f"{col}: final missing positions differ "
            "from source XPT."
        )


print(
    "PASS: Missing positions preserved."
)


# =================================================================================================
# 46. FINAL CDC CODE DISTRIBUTION CHECKS
# =================================================================================================

EXPECTED_FINAL_DISTRIBUTIONS = {

    "SLQ030": {

        "0": 1688,

        "1": 1370,

        "2": 1068,

        "3": 1577,

        "7": 7,

        "9": 451

    },


    "SLQ040": {

        "0": 4474,

        "1": 699,

        "2": 361,

        "3": 280,

        "7": 3,

        "9": 344

    },


    "SLQ050": {

        "1": 1621,

        "2": 4535,

        "9": 5

    },


    "SLQ120": {

        "0": 1028,

        "1": 1468,

        "2": 2059,

        "3": 1066,

        "4": 528,

        "9": 12

    }

}


for col, expected in EXPECTED_FINAL_DISTRIBUTIONS.items():

    actual = (

        raw_csv[col]

        .value_counts(
            dropna=False
        )

        .to_dict()

    )


    if actual != expected:

        raise ValueError(
            f"Final {col} distribution differs.\n\n"
            f"Expected:\n{expected}\n\n"
            f"Actual:\n{actual}"
        )


    print(
        f"PASS: Final {col} matches CDC."
    )


# =================================================================================================
# 47. FINAL CHARACTER FREQUENCY RECHECK
# =================================================================================================

FINAL_CHARACTER_EXPECTATIONS = {

    "SLQ300": {

        "recorded": 6087,

        "refused": 1,

        "dontknow": 36,

        "missing": 37

    },


    "SLQ310": {

        "recorded": 6100,

        "refused": 1,

        "dontknow": 21,

        "missing": 39

    },


    "SLQ320": {

        "recorded": 6090,

        "refused": 1,

        "dontknow": 39,

        "missing": 31

    },


    "SLQ330": {

        "recorded": 6102,

        "refused": 1,

        "dontknow": 26,

        "missing": 32

    }

}


for col, expected in FINAL_CHARACTER_EXPECTATIONS.items():

    text = raw_csv[col]


    recorded = int(

        text.str.fullmatch(
            r"\d{2}:\d{2}",
            na=False
        )

        .sum()

    )


    refused = int(

        text.eq(
            "77777"
        )

        .sum()

    )


    dontknow = int(

        text.eq(
            "99999"
        )

        .sum()

    )


    missing = int(

        text.eq("")
        .sum()

    )


    if recorded != expected["recorded"]:

        raise ValueError(
            f"Final {col} recorded-time count differs."
        )


    if refused != expected["refused"]:

        raise ValueError(
            f"Final {col} Refused count differs."
        )


    if dontknow != expected["dontknow"]:

        raise ValueError(
            f"Final {col} Don't know count differs."
        )


    if missing != expected["missing"]:

        raise ValueError(
            f"Final {col} missing count differs."
        )


    print(
        f"PASS: Final {col} matches CDC."
    )


# =================================================================================================
# 48. FINAL SLD012 / SLD013 CDC RECHECK
# =================================================================================================

final_sld012 = pd.to_numeric(

    raw_csv["SLD012"]
    .where(
        raw_csv["SLD012"] != "",
        np.nan
    ),

    errors="raise"

)


final_sld013 = pd.to_numeric(

    raw_csv["SLD013"]
    .where(
        raw_csv["SLD013"] != "",
        np.nan
    ),

    errors="raise"

)


if int(
    final_sld012.notna().sum()
) != 6113:

    raise ValueError(
        "Final SLD012 populated count differs."
    )


if int(
    final_sld013.notna().sum()
) != 6104:

    raise ValueError(
        "Final SLD013 populated count differs."
    )


if float(
    final_sld012.min()
) != 2.0:

    raise ValueError(
        "Final SLD012 minimum differs."
    )


if float(
    final_sld012.max()
) != 14.0:

    raise ValueError(
        "Final SLD012 maximum differs."
    )


if float(
    final_sld013.min()
) != 2.0:

    raise ValueError(
        "Final SLD013 minimum differs."
    )


if float(
    final_sld013.max()
) != 14.0:

    raise ValueError(
        "Final SLD013 maximum differs."
    )


print(
    "PASS: Final SLD012 / SLD013 ranges and counts validated."
)


# =================================================================================================
# 49. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 SLQ_J CONVERSION COMPLETE"
)


print(
    "=" * 110
)


print(
    "Component:"
)


print(
    "  Questionnaire"
)


print(
    ""
)


print(
    "Section:"
)


print(
    "  Sleep Disorders"
)


print(
    ""
)


print(
    "Cycle:"
)


print(
    "  2017-2018"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(raw_csv):,}"
)


print(
    "Columns:",
    len(raw_csv.columns)
)


print(
    "Unique SEQN:",
    f"{raw_csv['SEQN'].nunique():,}"
)


print(
    "SEQN range:"
)


print(
    "  93705-102956"
)


print(
    ""
)


print(
    "Validated XPORT zero restorations:"
)


print(
    "  SLQ030 = 1,688"
)


print(
    "  SLQ040 = 4,474"
)


print(
    "  SLQ120 = 1,028"
)


print(
    "  ----------------"
)


print(
    "  TOTAL   = 7,190"
)


print(
    ""
)


print(
    "Character variables preserved text-for-text:"
)


print(
    "  SLQ300"
)


print(
    "  SLQ310"
)


print(
    "  SLQ320"
)


print(
    "  SLQ330"
)


print(
    ""
)


print(
    "Genuine decimal variables preserved:"
)


print(
    "  SLD012"
)


print(
    "    Populated = 6,113"
)


print(
    "    Missing   = 48"
)


print(
    "    Range     = 2.0-14.0"
)


print(
    "    Fractional observations = 1,833"
)


print(
    ""
)


print(
    "  SLD013"
)


print(
    "    Populated = 6,104"
)


print(
    "    Missing   = 57"
)


print(
    "    Range     = 2.0-14.0"
)


print(
    "    Fractional observations = 1,337"
)


print(
    ""
)


print(
    "SLQ030=0 / Never:"
)


print(
    "  1,688"
)


print(
    "SLQ040=0 / Never:"
)


print(
    "  4,474"
)


print(
    "SLQ120=0 / Never:"
)


print(
    "  1,028"
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Remaining tiny XPORT artifacts: 0"
)


print(
    "Character-value changes: 0"
)


print(
    "SLD012 integerization: 0"
)


print(
    "SLD013 integerization: 0"
)


print(
    "Missing-position changes: 0"
)


print(
    "Scientific notation: 0"
)


print(
    "Numeric XPT -> CSV exact fidelity: PASS"
)


print(
    "Character source -> CSV fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2017-2018 SLQ_J
SLEEP DISORDERS QUESTIONNAIRE
Rows: 6,161
Columns: 11
Variables:
['SEQN', 'SLQ300', 'SLQ310', 'SLD012', 'SLQ320', 'SLQ330', 'SLD013', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']

Pandas source dtypes:
SEQN      float64
SLQ300        str
SLQ310        str
SLD012    float64
SLQ320        str
SLQ330        str
SLD013    float64
SLQ030    float64
SLQ040    float64
SLQ050    float64
SLQ120    float64
dtype: object
PASS: SLQ_J structure = 6,161 rows x 11 columns.

--- FIELD TYPE PLAN ---
Character fields: ['SLQ300', 'SLQ310', 'SLQ320', 'SLQ330']
Genuine decimal fields: ['SLD012', 'SLD013']
Integral/code fields: ['SEQN', 'SLQ030', 'SLQ040', 'SLQ050', 'SLQ120']
PASS: Numeric source fields validated.
PASS: SLQ300 character content validated.
PASS: SLQ310 character content validated.
PASS: SLQ320 character content validated.
PASS: SLQ330 character content validated.

--- SEQN SOURCE CHECK ---
Minimum: 93705
Maximum: 102956
Missing: 0
Duplicates: 0
PASS: SEQN range 93705-102956